# MDR-TS v19.2
**Temporal-Station Baseline Model for Soil Moisture Prediction**

**Author:** Jakob Balkovec  
**Affiliation:** Seattle University, Computer Science  
**Project:** MDR
**Notebook Type:** Training & Evaluation  
**Last Updated:** Tue Jan 6th 2026

---

## Model Summary
- **Model Name:** MDR-TS  
- **Version:** v19.2
- **Task:** Regression (Soil Moisture at 5 cm depth)  
- **Target Variable:** `soil_moisture_5cm`  
- **Temporal Resolution:** Daily  

---

## Reproducibility
- **Random Seed:** 42
- **Split Metadata:** `data/splits/derived_8.0/split_meta.json`
- **Environment:** Google Colab / VS Code Remote Kernel

---

Adapted for a Macbook M2 Pro environment.

**What's new?**

- Playing around with feature sets

## 0. Imports

In [ ]:
import os
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from xgboost import XGBRegressor

import torch

project_root = os.path.abspath("../../")
if project_root not in sys.path:
    sys.path.append(project_root)

from Utils.dashboard import metrics_dashboard

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("imports loaded")
print(f"using: {device}")

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

In [ ]:
SEED = 42
DEEP_SEARCH = 40

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"Random seed set to {SEED}")

def print_env_info():
    print("Environment information:")
    print(f"  Python version: {os.sys.version.split()[0]}")
    print(f"  NumPy version:  {np.__version__}")
    print(f"  Pandas version: {pd.__version__}")

    try:
        import xgboost
        print(f"  XGBoost version: {xgboost.__version__}")
    except ImportError:
        print("  XGBoost not installed")

    IN_COLAB = "COLAB_GPU" in os.environ
    print(f"  Running in Colab: {IN_COLAB}")

    if IN_COLAB:
        gpu = os.environ.get("COLAB_GPU", None)
        print(f"  GPU available: {gpu}")
    else:
        print("  GPU available: False")

print_env_info()

plt.style.use("default")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

print("environment setup complete")

In [ ]:
VERSION = "v19"
SUBVERSION = "v19.2"
RUN_NAME = "mdr_ts_v19_2"

def find_project_root():
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for cand in candidates:
        if (cand / "Temporal/Pipeline/data").exists() and (cand / "Models/Temporal").exists():
            return cand
    raise FileNotFoundError("Could not locate repo root containing Temporal/Pipeline/data")

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "Temporal/Pipeline/data"
SPLIT_ROOT = DATA_ROOT / "splits"
OUTPUT_ROOT = PROJECT_ROOT / "Models/Temporal" / VERSION / SUBVERSION

os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Project paths:")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  DATA_ROOT:    {DATA_ROOT}")
print(f"  SPLIT_ROOT:   {SPLIT_ROOT}")
print(f"  OUTPUT_ROOT:  {OUTPUT_ROOT}")

print("\nKey file checks:")
print("  data exists:", DATA_ROOT.exists())
print("  splits exists:", SPLIT_ROOT.exists())
print("  output exists:", OUTPUT_ROOT.exists())


In [ ]:
TRAIN_PATH = str(Path(SPLIT_ROOT) / "derived_8.0/train.csv")
VAL_PATH   = str(Path(SPLIT_ROOT) / "derived_8.0/val.csv")
TEST_PATH  = str(Path(SPLIT_ROOT) / "derived_8.0/test.csv")

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing split file: {p}")

print("Split files:")
print(" ", TRAIN_PATH)
print(" ", VAL_PATH)
print(" ", TEST_PATH)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

In [ ]:
print("Common columns across splits:",
      len(set(train_df.columns) & set(val_df.columns) & set(test_df.columns)))

print("\ncolumns:")
print(list(train_df.columns)[:30])

In [ ]:
TARGET_COL = "soil_moisture_5cm"

KEEP_META_COLS = ["station_id", "date", "longitude", "latitude"]

FEATURE_COLS = [
    'SMAP_sm_pm_interp_ema02',
    'V_rollmin_LST_modis_kobs30',
    'D_sin_DOY', 'G_rain_sum_3d',
    'V_ema_G_API_kobs7',
    'V_rollmin_G_API_kobs30',
    'G_rain_sum_7d',
    'C_lag_LST_modis_kobs30',
    'C_lag_G_API_kobs1',
    'V_ema_G_API_kobs14',
    'V_rollmean_G_API_kobs14',
    'G_API', 'G_DSLR',
    'SMAP_ampm_diff_interp',
    'V_rollmax_G_API_kobs30',
    'V_ema_G_API_kobs30',
    'V_rollmean_s2_b11_kobs7',
    'V_ema_LST_modis_kobs7',
    'V_rollmean_G_API_kobs7',
    'C_lag_s2_b11_kobs30',
    'A_d_E_SAR_diff_kobs14',
    'C_lag_LST_modis_kobs6',
    'A_d_LST_modis_kobs14',
    'A_d_SMAP_sm_interp_kobs14',
    'V_rollstd_SMAP_sm_interp_kobs30',
    'SMAP_sm_interp_grad7',
    'year_frac', 'sin_year', 'cos_year',
    'API_x_year', 'SMAP_x_year',
    'slope', 'elev', 'K_slope_sin',
    'K_slope_cos', 'K_aspect_cos',
    'J_clay_wfrac_b0', 'J_sand_wfrac_b0'
    ]

expected = set(KEEP_META_COLS + FEATURE_COLS + [TARGET_COL])
missing_train = sorted(list(expected - set(train_df.columns)))
missing_val   = sorted(list(expected - set(val_df.columns)))
missing_test  = sorted(list(expected - set(test_df.columns)))

if missing_train or missing_val or missing_test:
    raise ValueError(
        f"Missing columns:\n"
        f"  train: {missing_train}\n"
        f"  val:   {missing_val}\n"
        f"  test:  {missing_test}"
    )

print("Columns locked")
print("  Features:", len(FEATURE_COLS))
print("  Target:  ", TARGET_COL)

In [ ]:
corr = train_df[FEATURE_COLS].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

# find pairs > 0.995 correlation
high_corr = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if upper.loc[row, col] > 0.995
]

print("Highly correlated pairs:")
for a, b, c in high_corr:
    print(f"{a} <-> {b} : {c:.5f}")

In [ ]:
def get_metrics_dict(y_true, y_pred, prefix=""):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    err = y_true - y_pred
    ae = np.abs(err)

    r2 = float(r2_score(y_true, y_pred))
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(root_mean_squared_error(y_true, y_pred))

    bias = float(np.mean(err))
    ubrmse = float(np.std(err))

    q_err = np.quantile(err, [0.05, 0.25, 0.50, 0.75, 0.95])

    return {
        f"{prefix}n": int(len(y_true)),
        f"{prefix}r2": r2,
        f"{prefix}mae": mae,
        f"{prefix}rmse": rmse,
        f"{prefix}ubrmse": ubrmse,
        f"{prefix}bias": bias,
        f"{prefix}med_ae": float(np.median(ae)),
        f"{prefix}p90_ae": float(np.quantile(ae, 0.90)),
        f"{prefix}q05_err": float(q_err[0]),
        f"{prefix}q50_err": float(q_err[2]),
        f"{prefix}q95_err": float(q_err[4]),
    }

def neat_print(metrics):
    print(f"{'METRIC':<15} | {'VALUE':<10}")
    print("-" * 28)
    for k, v in metrics.items():
        val_str = f"{v:,}" if isinstance(v, int) else f"{v:+.5f}"
        print(f"{k:<15} | {val_str:<10}")

### Split Strategy
- **Training set:**  
  Two stations, early time period  
- **Validation set:**  
  Same stations as training, held-out **future dates** (temporal holdout)
- **Test set:**  
  One completely unseen station (station-level holdout)

### Motivation
- Validation evaluates **temporal generalization** on known stations
- Test evaluates **spatial generalization** to an unseen station
- This avoids spatial leakage while preserving sufficient training data

In [ ]:
print("=== SPLIT SUMMARY ===")

def split_summary(name, d):
    print(f"\n{name.upper()}")
    print(f"  rows:     {len(d)}")
    print(f"  stations: {sorted(d['station_id'].unique().tolist())}")
    if "date" in d.columns:
        print(f"  date range: {d['date'].min()} -- {d['date'].max()}")

split_summary("train", train_df)
split_summary("val", val_df)
split_summary("test", test_df)

print("\n=== LEAKAGE CHECK ===")
print("train ∩ test:", sorted(set(train_df.station_id) & set(test_df.station_id)))
print("val   ∩ test:", sorted(set(val_df.station_id) & set(test_df.station_id)))

print("\n-- split locked --")


In [ ]:
trainval_df_d = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)

trainval_df_d["date"] = pd.to_datetime(trainval_df_d["date"], errors="coerce")
trainval_df_d["year"] = trainval_df_d["date"].dt.year.astype(float)

max_year = trainval_df_d["year"].max()
beta = 0.2

w_trainval = np.exp(beta * (trainval_df_d["year"] - max_year))
w_trainval = w_trainval / w_trainval.mean()

In [ ]:
trainval_df_d = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)

X_trainval_d = trainval_df_d[FEATURE_COLS].copy()
y_trainval_d = trainval_df_d[TARGET_COL].copy()

X_test_d = test_df[FEATURE_COLS].copy()
y_test_d = test_df[TARGET_COL].copy()

print("\nDRIFT matrices:")
print("  X_trainval_d:", X_trainval_d.shape)
print("  X_test_d:    ", X_test_d.shape)

### Drift Model (No Weights)

In [ ]:
XGB_PARAMS_DRIFT_NO_W = dict(
    objective="reg:absoluteerror",
    random_state=SEED,
    n_jobs=-1,
    subsample=0.9,
    colsample_bytree=0.8,
    max_depth=8,
    min_child_weight=2,
    n_estimators=5500,
    learning_rate=0.04,
    reg_lambda=1.5,
    reg_alpha=0.03,
    gamma=0.0,
)

In [ ]:
trainval_df = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)

xgb_drift_no_w_tv = XGBRegressor(**XGB_PARAMS_DRIFT_NO_W)
xgb_drift_no_w_tv.fit(
  trainval_df_d[FEATURE_COLS],
  trainval_df_d[TARGET_COL],
  verbose=0,
  )

y_test_d_no_w = np.asarray(test_df[TARGET_COL]).ravel()
pred_drift_test_no_w = np.asarray(xgb_drift_no_w_tv.predict(test_df[FEATURE_COLS])).ravel()

In [ ]:
metrics_dashboard(y_test_d_no_w, pred_drift_test_no_w, name="Test No Weights", return_dict=False)

In [ ]:
print("===== NON-WEIGHTED MODEL METRICS =====")
neat_print(get_metrics_dict(y_test_d, pred_drift_test_no_w, prefix="no_w_"))

### Drift Model (With Weights)

In [ ]:
XGB_PARAMS_DRIFT_W = dict(
    objective="reg:pseudohubererror",
    random_state=SEED,
    n_jobs=-1,
    subsample=0.9,
    colsample_bytree=0.8,
    max_depth=8,
    min_child_weight=2,
    n_estimators=5500,
    learning_rate=0.04,
    reg_lambda=1.5,
    reg_alpha=0.03,
    gamma=0.0,
)

In [ ]:
w_trainval_s = pd.Series(w_trainval)
years_tv = trainval_df_d["year"].reset_index(drop=True)

print("Weight stats:")
print(w_trainval_s.describe())

min_year = years_tv.min()
max_year = years_tv.max()

print("Min year weight:", float(w_trainval_s[years_tv == min_year].mean()))
print("Max year weight:", float(w_trainval_s[years_tv == max_year].mean()))

In [ ]:
trainval_df_d = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)

xgb_drift_w_tv = XGBRegressor(**XGB_PARAMS_DRIFT_W)
xgb_drift_w_tv.fit(
  trainval_df_d[FEATURE_COLS],
  trainval_df_d[TARGET_COL],
  verbose=0,
  sample_weight=w_trainval
  )

y_test_d_w = np.asarray(test_df[TARGET_COL]).ravel()
pred_drift_test_w = np.asarray(xgb_drift_w_tv.predict(test_df[FEATURE_COLS])).ravel()

In [ ]:
metrics_dashboard(y_test_d_w, pred_drift_test_w, name="Test w Weights", return_dict=False)

In [ ]:
print("======= WEIGHTED MODEL METRICS =======")
neat_print(get_metrics_dict(y_test_d, pred_drift_test_w, prefix="no_w_"))

In [ ]:
diag_d = test_df.copy().reset_index(drop=True)
diag_d["year"] = pd.to_datetime(diag_d["date"], errors="coerce").dt.year
diag_d["y"] = y_test_d
diag_d["pred"] = pred_drift_test_w

d2023 = diag_d[diag_d["year"] == 2023].copy()

if len(d2023) >= 2:
    r2_2023 = float(r2_score(d2023["y"], d2023["pred"]))
    a, b = np.polyfit(d2023["pred"], d2023["y"], 1)  # y ≈ a*pred + b
    print("\nDRIFT 2023 R2:", r2_2023)
    print("DRIFT 2023 slope:", float(a), "intercept:", float(b))
else:
    print("\nNo 2023 rows found in DRIFT test slice.")

In [ ]:
def get_xgb_importance(model, feature_cols):
    imp = pd.Series(model.feature_importances_, index=feature_cols)
    imp = imp / imp.sum()
    return imp

def get_rf_importance(model, feature_cols):
    rf_model = model.named_steps["model"] if hasattr(model, "named_steps") else model
    imp = pd.Series(rf_model.feature_importances_, index=feature_cols)
    imp = imp / imp.sum()
    return imp

imp_no_w = get_xgb_importance(xgb_drift_no_w_tv, FEATURE_COLS)
imp_w    = get_xgb_importance(xgb_drift_w_tv, FEATURE_COLS)

all_feats = sorted(set(imp_no_w.index) | set(imp_w.index))

cmp = pd.DataFrame({
    "drift_no_w": imp_no_w.reindex(all_feats, fill_value=0.0),
    "drift_w":    imp_w.reindex(all_feats, fill_value=0.0),
})

cmp["consensus_mean"] = cmp.mean(axis=1)

TOP_K = 30
cmp_top = cmp.sort_values("consensus_mean", ascending=False).head(TOP_K)

cmp_top_sorted = cmp_top.sort_values("consensus_mean", ascending=True)
plot_df = cmp_top_sorted[["drift_no_w", "drift_w"]]

plt.figure(figsize=(10, 8))
y = np.arange(len(plot_df))

plt.barh(y - 0.25, plot_df["drift_no_w"].values, height=0.25, label="Drift (no W)")
plt.barh(y,         plot_df["drift_w"].values,    height=0.25, label="Drift (β=0.2)")

plt.yticks(y, plot_df.index)
plt.xlabel("Normalized Importance (sum = 1 per model)")
plt.title(f"Top {TOP_K} Feature Importances (2-Model Comparison)")
plt.grid(axis="x", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
display(cmp_top)

### Residuals

In [ ]:
res_w   = y_test_d - pred_drift_test_w
res_nw  = y_test_d - pred_drift_test_no_w

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

axes[0].scatter(y_test_d, res_w, s=8, alpha=0.6)
axes[0].axhline(0)
axes[0].set_xlabel("True Soil Moisture")
axes[0].set_ylabel("Residual (true - pred)")
axes[0].set_title("Weighted")

axes[1].scatter(y_test_d, res_nw, s=8, alpha=0.6)
axes[1].axhline(0)
axes[1].set_xlabel("True Soil Moisture")
axes[1].set_title("No Weights")

axes[2].scatter(y_test_d, res_w,  s=8, alpha=0.5, label="Weighted")
axes[2].scatter(y_test_d, res_nw, s=8, alpha=0.5, label="No Weights")
axes[2].axhline(0)
axes[2].set_xlabel("True Soil Moisture")
axes[2].set_title("Overlay")
axes[2].legend()

plt.tight_layout()
plt.show()

---

_Jakob Balkovec_

In [ ]:
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

X_shap = X_test_d.sample(n=min(1000, len(X_test_d)), random_state=42).copy()
dmatrix_shap = xgb.DMatrix(X_shap, feature_names=FEATURE_COLS)

booster = xgb_drift_w_tv.get_booster()
contribs = booster.predict(dmatrix_shap, pred_contribs=True)
shap_values = contribs[:, :-1]

plt.figure()
shap.summary_plot(
    shap_values,
    X_shap,
    feature_names=FEATURE_COLS,
    max_display=100,
    show=False
)
plt.tight_layout()
plt.show()

mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_idx = np.argsort(mean_abs_shap)[::-1][:10]
top_features = pd.DataFrame({
    "feature": np.array(FEATURE_COLS)[top_idx],
    "mean_abs_shap": mean_abs_shap[top_idx]
})
print(top_features.to_string(index=False))

## Nash-Sutcliffe Efficiency (NSE) Calculation

In [27]:
def calculate_nse(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    numerator = np.sum((y_true - y_pred)**2)
    denominator = np.sum((y_true - np.mean(y_true))**2)

    return 1 - (numerator / denominator)

nse_value_w = calculate_nse(y_test_d, pred_drift_test_w)
print(f"NSE for the Weighted Model: {nse_value_w:.5f}")

nse_value_no_w = calculate_nse(y_test_d, pred_drift_test_no_w)
print(f"NSE for the non-Weighted Model: {nse_value_w:.5f}")

NSE for the Weighted Model: 0.82239
NSE for the non-Weighted Model: 0.82239
